<a href="https://colab.research.google.com/github/ChadapaNgamchuen/AI-pokemon-color-matcher/blob/main/pokemon_color_ClassRoom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project title
Pokemon Color Matcher
This project uses DSPy and the Gemini API to find a Pokemon that best
matches a given color combination — demonstrating how to structure
LLM outputs into reliable, typed fields.

# Summary

This project demonstrates the core DSPy workflow taught in class:

  1. Signature: Define the task as typed input/output fields
  2. ChainOfThought: Add step-by-step reasoning to improve accuracy
  3. Function: Wrap the predictor and return a dictionary
  4. Batch loop: Apply the same function across a list of inputs
  5. DataFrame : Collect and display results in a structured table

The same pattern was applied to the AI News Summarizer project,
replacing color themes with news articles and Pokemon names with
summaries, insights, sentiment, and tags.

This shows how one reusable pattern can solve very different problems
simply by changing the Signature definition.

### Step 1: Setup Model

We install DSPy and configure Gemini 2.5 Pro as our language model engine.

  - dspy.LM        : Connects to the Gemini API
  - dspy.configure : Sets the model globally so every DSPy module uses it
                     automatically — no need to pass the model each time

Key concept learned in class:
DSPy separates "what model to use" from "what task to run", making it
easy to swap models without rewriting any logic.

---

### Step 2: Define the Task (Signature)

A DSPy Signature is like a function contract — it declares exactly what
goes in and what comes out, without writing a prompt manually.

* Input:
    - input_color  : A color description, e.g., "orange and cyan"

* Outputs:
    - pokemon_name : The name of the best-matching Pokemon
    - ability      : A description of that Pokemon's abilities

In [ ]:
pip install -q dspy-ai

In [ ]:
import dspy
lm = dspy.LM("gemini/gemini-2.5-pro", api_key="")
dspy.configure(lm=lm)

In [ ]:
# 2. Define the DSPy Signature using modern type hints
class PokemonColorMatcher(dspy.Signature):
    """You are a Pokemon expert. Find a Pokemon pet that closely matches the given input colors."""

    input_color: str = dspy.InputField(desc="The colors of the Pokemon, e.g., 'magenta and neo-green' or 'orange and cyan'")
    pokemon_name: str = dspy.OutputField(desc="The name of the matching Pokemon")
    ability: str = dspy.OutputField(desc="A list and description of the Pokemon's abilities")

# 3. Create the DSPy Module
# ChainOfThought allows the model to reason about the colors before answering
pokemon_predictor = dspy.ChainOfThought(PokemonColorMatcher)

# 4. Execute the Task
def get_pokemon_by_color(color: str):
    print(f"Searching for a {color} Pokémon...")

    # Call the predictor with our input
    result = pokemon_predictor(input_color=color)

    # Print the formatted output
    print("-" * 30)
    print(f"Pokemon Name: {result.pokemon_name}")
    print(f"Ability:\n{result.ability}")
    print("-" * 30)
    print(f"Thought:\n{result.reasoning}")

### Step 3: Test with a Single Query

Before processing a large list, we test the pipeline with one color
at a time to verify the output is correct.

dspy.ChainOfThought wraps our Signature and adds a reasoning step —
the model thinks through which Pokemon fits the colors before answering.
This generally produces more accurate results than asking directly.

In [ ]:
get_pokemon_by_color("magenta and neo-green")

Searching for a magenta and neo-green Pokémon...
------------------------------
Pokemon Name: Noivern
Ability:
*   **Frisk:** When Noivern enters a battle, it can check an opposing Pokémon's held item.
*   **Infiltrator:** This ability allows Noivern's attacks to bypass the effects of the target's protective moves like Reflect, Light Screen, and Substitute.
*   **Telepathy (Hidden Ability):** In Double or Triple Battles, Noivern anticipates and dodges attacks from its own allies.
------------------------------
Thought:
Noivern is a perfect match for the colors magenta and neo-green. This large, bat-like Pokémon has a sleek, dark grey and black body, which creates a stunning contrast for its more vibrant features. The entire inner membrane of its massive wings and its large ears are a brilliant, striking green that fits the description of "neo-green." Complementing this is the fluffy mane around its neck and its underbelly, which are a distinct and rich purplish-magenta. The combination

In [ ]:
get_pokemon_by_color("yellow and black")

Searching for a yellow and black Pokémon...
------------------------------
Pokemon Name: Beedrill
Ability:
*   **Swarm:** When the Pokémon's HP is at 1/3 or less of its maximum, the power of its Bug-type moves is increased by 50%.
*   **Sniper (Hidden Ability):** Powers up the damage of critical hits. Instead of the usual 1.5x damage, critical hits will do 2.25x damage.
------------------------------
Thought:
The colors yellow and black are a classic warning combination found in nature, often associated with stinging insects. The Pokémon that perfectly embodies this is Beedrill. Known as the Poison Bee Pokémon, its design is a direct reflection of these colors. Its body is a vibrant yellow, sharply contrasted by distinct black stripes and black limbs. This striking pattern emphasizes its aggressive nature and the powerful stingers on its forelimbs and abdomen, making it an unmistakable match for a yellow and black theme.


### Step 4: Define Color Themes

We define 20 color combination strings as our input dataset.
Each string describes two colors that a Pokemon might have,
ranging from common combinations like "Yellow and Black"
to creative ones like "Vanilla and Matcha".

---

### Step 5: Batch Processing with a Loop

We iterate through every color theme, call get_pokemon_by_color(),
and collect each result dictionary into a list.

This is the core pattern for batch AI processing:

```
results = []
  for item in input_list:
      results.append(model(item)) ← same structure every time
```
Key concept learned in class:
The function returns a dictionary with fixed keys ("Color", "Name", "Ability").
Keeping the output structure consistent is what makes it possible to
convert the list directly into a DataFrame in the next step.

---

### Step 6: Display Results as a Table

We convert the list of dictionaries into a pandas DataFrame
and select the column order we want: Color > Name > Ability.

In [ ]:
import pandas as pd

# Setup model
lm = dspy.LM("gemini/gemini-2.5-pro", api_key="")
dspy.configure(lm=lm)

# 1. Define the DSPy Signature
class PokemonColorMatcher(dspy.Signature):
    """You are a Pokemon expert. Find a Pokemon pet that closely matches the given input colors."""
    input_color: str = dspy.InputField(desc="The colors of the Pokemon, e.g., 'magenta and neo-green'")
    pokemon_name: str = dspy.OutputField(desc="The name of the matching Pokemon")
    ability: str = dspy.OutputField(desc="A list and description of the Pokemon's abilities")

pokemon_predictor = dspy.ChainOfThought(PokemonColorMatcher)

# 2. แก้ไขฟังก์ชันให้ Return ค่าเป็น Dictionary ที่มี Key ตามที่ต้องการ
def get_pokemon_by_color(color: str):
    print(f"กำลังค้นหาโปเกมอนสำหรับสี: {color}...")

    # ดึงผลลัพธ์จาก DSPy
    result = pokemon_predictor(input_color=color)

    # สำคัญมาก: ต้อง return ข้อมูลกลับไปเพื่อนำไปสร้างตาราง
    return {
        "Color": color,
        "Name": result.pokemon_name,
        "Ability": result.ability
    }

# รายชื่อสี
color_themes = [
    "Yellow and Black", "Green and Magenta", "Orange and Cyan",
    "Teal and Coral", "Plum and Chartreuse", "Navy and Gold",
    "Indigo and Peach", "Fuchsia and Lime", "Mustard and Charcoal",
    "Olive and Blush", "Copper and Slate", "Terracotta and Sage",
    "Rust and Denim", "Mint and Lavender", "Crimson and Aqua",
    "Ruby and Emerald", "Tangerine and Periwinkle",
    "Rose Gold and Dusty Blue", "Lilac and Champagne", "Vanilla and Matcha"
]

# 3. วนลูปเพื่อเก็บข้อมูล
all_pokemon_data = []

for theme in color_themes:
    # เรียกใช้ฟังก์ชันและเก็บค่า Dictionary ที่ Return กลับมา
    result_dict = get_pokemon_by_color(theme)
    all_pokemon_data.append(result_dict)

# 4. แปลงเป็น DataFrame และจัดเรียงคอลัมน์
df = pd.DataFrame(all_pokemon_data)
df = df[["Color", "Name", "Ability"]]

df

กำลังค้นหาโปเกมอนสำหรับสี: Yellow and Black...
กำลังค้นหาโปเกมอนสำหรับสี: Green and Magenta...
กำลังค้นหาโปเกมอนสำหรับสี: Orange and Cyan...
กำลังค้นหาโปเกมอนสำหรับสี: Teal and Coral...
กำลังค้นหาโปเกมอนสำหรับสี: Plum and Chartreuse...
กำลังค้นหาโปเกมอนสำหรับสี: Navy and Gold...
กำลังค้นหาโปเกมอนสำหรับสี: Indigo and Peach...
กำลังค้นหาโปเกมอนสำหรับสี: Fuchsia and Lime...
กำลังค้นหาโปเกมอนสำหรับสี: Mustard and Charcoal...
กำลังค้นหาโปเกมอนสำหรับสี: Olive and Blush...
กำลังค้นหาโปเกมอนสำหรับสี: Copper and Slate...
กำลังค้นหาโปเกมอนสำหรับสี: Terracotta and Sage...
กำลังค้นหาโปเกมอนสำหรับสี: Rust and Denim...
กำลังค้นหาโปเกมอนสำหรับสี: Mint and Lavender...
กำลังค้นหาโปเกมอนสำหรับสี: Crimson and Aqua...
กำลังค้นหาโปเกมอนสำหรับสี: Ruby and Emerald...
กำลังค้นหาโปเกมอนสำหรับสี: Tangerine and Periwinkle...
กำลังค้นหาโปเกมอนสำหรับสี: Rose Gold and Dusty Blue...
กำลังค้นหาโปเกมอนสำหรับสี: Lilac and Champagne...
กำลังค้นหาโปเกมอนสำหรับสี: Vanilla and Matcha...


,Color,Name,Ability
0,Yellow and Black,Beedrill,* **Swarm:** When Beedrill's HP is low (belo...
1,Green and Magenta,Noivern,* **Frisk:** When the Pokémon enters a battl...
2,Orange and Cyan,Volcarona,Flame Body: When this Pokemon is hit by a move...
3,Teal and Coral,Milotic,1. Marvel Scale: If this Pokémon is afflicted ...
4,Plum and Chartreuse,Dragapult,1. **Clear Body**: Prevents other Pokémon fro...
5,Navy and Gold,Empoleon,**Torrent:** Powers up Water-type moves by 50%...
6,Indigo and Peach,Mawile,1. **Hyper Cutter**: Prevents other Pokémon fr...
7,Fuchsia and Lime,Goodra,1. Sap Sipper: When this Pokemon is hit by a G...
8,Mustard and Charcoal,Umbreon,* **Synchronize:** When this Pokemon is burn...
9,Olive and Blush,Virizion,Justified: Boosts the Attack stat by one stage...
